<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/01)_Scaled_Dot_product_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결 및 프로젝트 경로 설정

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/attention_is_all_you_need")
SRC_DIR = PROJECT_ROOT / "src"

# 폴더가 없으면 생성
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

# 프로젝트 루트로 이동
%cd /content/drive/MyDrive/attention_is_all_you_need

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SRC_DIR      :", SRC_DIR)

Mounted at /content/drive
/content/drive/MyDrive/attention_is_all_you_need
PROJECT_ROOT : /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR      : /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 필요한 라이브러리 불러오기

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [ ]:
# Shape 잡고가기
'''
Q : (batch_size, query_len, d_k)
K : (batch_size, key_len,   d_k)
V : (batch_size, value_len,   d_v)

인 경우,

계산을 진행하면

Q @ K^T = (batch_size, query_len, d_k) @ (batch_size, d_k, key_len)
(이때 @는 행렬곱을 의미하며, 제일 뒤 2차원을 행렬곱한다.)

-> attention_weights = (batch_size, query_len, key_len)
(softmax, scaling, masking은 shape에 영향을 주지 않기에 생략)
이다.

따라서 attention_weight @ V를 진행하는 경우, V: (batch_size, value_len, d_v)가 된다.(value_len == key_len)

그럼으로 shape는
Q : (batch_size, query_len, d_k)
K : (batch_size, key_len,   d_k)
V : (batch_size, key_len,   d_v)

로 진행하게 된다
'''

'\nQ : (batch_size, query_len, d_k)\nK : (batch_size, key_len,   d_k)\nV : (batch_size, value_len,   d_v)\n\n인 경우,\n\n계산을 진행하면\n\nQ @ K^T = (batch_size, query_len, d_k) @ (batch_size, d_k, key_len)\n\n-> attention_weights = (batch_size, query_len, key_len)\n(softmax, scaling, masking은 shape에 영향을 주지 않기에 생략)\n이다.\n\n따라서 attention_weight @ V를 진행하는 경우, V: (batch_size, value_len, d_v)가 된다.(value_len == key_len)\n\n그럼으로 shape는\nQ : (batch_size, query_len, d_k)\nK : (batch_size, key_len,   d_k)\nV : (batch_size, key_len,   d_v)\n\n로 진행하게 된다\n'

In [ ]:
# Scaled Dot-Product Attention 구현


class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__() # nn.Module 클래스 상속

    def forward(self, Q, K, V, mask=None):
        """
        Q: (..., query_len, d_k)
        K: (..., key_len,   d_k)
        V: (..., key_len,   d_v)

        mask:
            attention을 허용할 위치 = True
            attention을 막을 위치 = False

        Returns:
            output:
                (..., query_len, d_v)

            attention_weights:
                (..., query_len, key_len)
        """

        # Q와 K의 마지막 차원 크기는 같아야 한다.
        if Q.size(-1) != K.size(-1): # 둘 다 차원이 d_k인지 확인
            raise ValueError(
                f"Q와 K의 d_k가 같아야 합니다. "
                f"Q: {Q.size(-1)}, K: {K.size(-1)}"
            )

        # K와 V는 같은 위치들을 나타내므로 sequence 길이가 같아야 한다.
        if K.size(-2) != V.size(-2): # cross attention이 아닌 경우에는 query_len과 key_len이 같다.
            raise ValueError(
                f"K와 V의 sequence 길이가 같아야 합니다. "
                f"K: {K.size(-2)}, V: {V.size(-2)}"
            )

        # d_k
        d_k = Q.size(-1)

        # 1. QK^T
        scores = Q @ K.transpose(-2, -1) # Q와 K를 계산 가능한 행렬로 바꿔줌(K를 전치함으로서)
        #scores: (..., query_len, key_len)

        # 2. sqrt(d_k)로 scaling
        scores = scores / math.sqrt(d_k) # 기울기 소모현상을 막기 위해 scaling

        # 3. 선택적으로 mask 적용
        if mask is not None:
            mask = mask.to(dtype=torch.bool, device=scores.device)

            # False 위치는 softmax 이후 확률이 0이 되도록 -inf 처리
            scores = scores.masked_fill(~mask, float("-inf"))

        '''
        mask를 적용하는 이유는 2가지가 있다.
        1) padding 처리를 하기 위해서: 문장을 bath로 묶어서 사용하는 경우 모든 문장의 길이를 맞춰줘야하는데, 이때 문장의 남는 공간을 0(Pad 토큰)으로 채운다. 이를 계산에서 제외하기 위해 masking 한다
        2) decoder에서 예측하고자 하는 위치 이후의 데이터를 참고하는 현상을 방지하기 위해
        '''


        # 4. key 방향으로 softmax
        attention_weights = F.softmax(scores, dim=-1)
        # 이를 통해 Q와 K를 비교하여 가중치를 얻는다

        # 5. Attention weight와 V를 행렬곱
        output = attention_weights @ V
        # V를 행렬곱하여 각각의 가중치만큼의 V가 Q를 설명한다.

        return output, attention_weights

In [ ]:
# 기본 동작 테스트

torch.manual_seed(42) # 난수 생성기 시드 고정

batch_size = 2
query_len = 3
key_len = 4
d_k = 5
d_v = 6

Q = torch.randn(batch_size, query_len, d_k)
K = torch.randn(batch_size, key_len, d_k)
V = torch.randn(batch_size, key_len, d_v)

attention = ScaledDotProductAttention()

output, attention_weights = attention(Q, K, V)

print("Q shape                 :", Q.shape)
print("K shape                 :", K.shape)
print("V shape                 :", V.shape)
print("attention_weights shape :", attention_weights.shape) # Q @ K^T
print("output shape            :", output.shape) # softmax((Q @ K^T)/d_k^(1/2)) @ V

Q shape                 : torch.Size([2, 3, 5])
K shape                 : torch.Size([2, 4, 5])
V shape                 : torch.Size([2, 4, 6])
attention_weights shape : torch.Size([2, 3, 4])
output shape            : torch.Size([2, 3, 6])


In [ ]:
# Attention 내부 Shape 확인

d_k = Q.size(-1)

K_transposed = K.transpose(-2, -1)
scores_before_scaling = Q @ K_transposed
scores_after_scaling = scores_before_scaling / math.sqrt(d_k) # d_k 제곱근
weights = F.softmax(scores_after_scaling, dim=-1)
manual_output = weights @ V

print("Q                       :", Q.shape)
print("K                       :", K.shape)
print("K.transpose(-2, -1)     :", K_transposed.shape)
print("Q @ K^T                 :", scores_before_scaling.shape)
print("Scaled scores           :", scores_after_scaling.shape)
print("Attention weights       :", weights.shape)
print("V                       :", V.shape)
print("Final output            :", manual_output.shape)

Q                       : torch.Size([2, 3, 5])
K                       : torch.Size([2, 4, 5])
K.transpose(-2, -1)     : torch.Size([2, 5, 4])
Q @ K^T                 : torch.Size([2, 3, 4])
Scaled scores           : torch.Size([2, 3, 4])
Attention weights       : torch.Size([2, 3, 4])
V                       : torch.Size([2, 4, 6])
Final output            : torch.Size([2, 3, 6])


In [ ]:
# 작은 숫자를 이용한 계산 검증

Q_small = torch.tensor([ # Q_small.shape = (1, 2, 2)
    [
        [1.0, 0.0],
        [0.0, 1.0]
    ]
])

K_small = torch.tensor([ # K_small.shape = (1, 2, 2)
    [
        [1.0, 0.0],
        [0.0, 1.0]
    ]
])

V_small = torch.tensor([ # V_small.shape = (1, 2, 2)
    [
        [10.0, 0.0],
        [0.0, 20.0]
    ]
])

print("Q:")
print(Q_small)

print("\nK:")
print(K_small)

print("\nV:")
print(V_small)

Q:
tensor([[[1., 0.],
         [0., 1.]]])

K:
tensor([[[1., 0.],
         [0., 1.]]])

V:
tensor([[[10.,  0.],
         [ 0., 20.]]])


In [ ]:
# QK^T 계산

K_T = K_small.transpose(-2, -1) # K_T.shape = (1, 2, 2)

scores = Q_small @ K_T # scores.shape = (1, 2, 2)

print("K^T:")
print(K_T)

print("\nQK^T:")
print(scores)

K^T:
tensor([[[1., 0.],
         [0., 1.]]])

QK^T:
tensor([[[1., 0.],
         [0., 1.]]])


In [ ]:
# Scaling

d_k_small = Q_small.size(-1)

scaled_scores = scores / math.sqrt(d_k_small) # scaling

print("d_k:", d_k_small)
print("sqrt(d_k):", math.sqrt(d_k_small))

print("\nScaled scores:")
print(scaled_scores)

d_k: 2
sqrt(d_k): 1.4142135623730951

Scaled scores:
tensor([[[0.7071, 0.0000],
         [0.0000, 0.7071]]])


In [ ]:
# Softmax

small_weights = F.softmax(scaled_scores, dim=-1)

print("Attention weights:")
print(small_weights)

print("\n각 Query별 weight 합:")
print(small_weights.sum(dim=-1))

Attention weights:
tensor([[[0.6698, 0.3302],
         [0.3302, 0.6698]]])

각 Query별 weight 합:
tensor([[1., 1.]])


In [ ]:
# 최종 Attention output

small_output = small_weights @ V_small

print("Final output:")
print(small_output)

Final output:
tensor([[[ 6.6976,  6.6048],
         [ 3.3024, 13.3952]]])


In [ ]:
# 클래스 결과와 수동 계산 결과 비교

class_output, class_weights = attention(
    Q_small,
    K_small,
    V_small
)

print("클래스 attention weights:")
print(class_weights)

print("\n직접 계산 attention weights:")
print(small_weights)

print("\n클래스 output:")
print(class_output)

print("\n직접 계산 output:")
print(small_output)

assert torch.allclose(class_weights, small_weights)
assert torch.allclose(class_output, small_output)

print("\n검증 성공")

클래스 attention weights:
tensor([[[0.6698, 0.3302],
         [0.3302, 0.6698]]])

직접 계산 attention weights:
tensor([[[0.6698, 0.3302],
         [0.3302, 0.6698]]])

클래스 output:
tensor([[[ 6.6976,  6.6048],
         [ 3.3024, 13.3952]]])

직접 계산 output:
tensor([[[ 6.6976,  6.6048],
         [ 3.3024, 13.3952]]])

검증 성공


In [ ]:
# 간단한 Mask 테스트
# Maks 규칙 : True -> 허용, False -> 차단
# masking 하는 경우 해당 인덱스를 -1e9로 채움(softmax는 e^x 를 사용하는데, x의 자리에 0이 들어가게 되면 1이 되기에 매우 작은 수인 -1e9를 사용한다.)

mask = torch.tensor([
    [
        [True, False],
        [True, True]
    ]
])

masked_output, masked_weights = attention(
    Q_small,
    K_small,
    V_small,
    mask=mask
)

print("Mask:")
print(mask)

print("\nMasked attention weights:")
print(masked_weights)

print("\nMasked output:")
print(masked_output)

Mask:
tensor([[[ True, False],
         [ True,  True]]])

Masked attention weights:
tensor([[[1.0000, 0.0000],
         [0.3302, 0.6698]]])

Masked output:
tensor([[[10.0000,  0.0000],
         [ 3.3024, 13.3952]]])


In [ ]:
# Maskes 위치가 0인지 검사

print(
    "가려진 위치의 attention weight:",
    masked_weights[0, 0, 1].item() # .item() tensor 를 scalar로 변환해서 반환
)

assert torch.isclose( # assert는 조건문 틀리면 오류 반환, isclose()는 두 원소가 같은지 여부 boolean 반환
    masked_weights[0, 0, 1],
    torch.tensor(0.0)
)

print("Mask 검증 성공")

가려진 위치의 attention weight: 0.0
Mask 검증 성공


In [ ]:
# 여러 Batch 입력 테스트

torch.manual_seed(0)

Q_batch = torch.randn(4, 3, 8) # 평균 0, 표준편차 1인 무작위 난수
K_batch = torch.randn(4, 5, 8)
V_batch = torch.randn(4, 5, 6)

batch_output, batch_weights = attention(
    Q_batch,
    K_batch,
    V_batch
)

print("Q shape      :", Q_batch.shape)
print("K shape      :", K_batch.shape)
print("V shape      :", V_batch.shape)

print()

print("Weights shape:", batch_weights.shape)
print("Output shape :", batch_output.shape)

Q shape      : torch.Size([4, 3, 8])
K shape      : torch.Size([4, 5, 8])
V shape      : torch.Size([4, 5, 6])

Weights shape: torch.Size([4, 3, 5])
Output shape : torch.Size([4, 3, 6])


In [ ]:
# 최종 동작 검증

torch.manual_seed(123)

attention = ScaledDotProductAttention()

# ---------------------------
# 기본 batch 입력
# ---------------------------

Q_test = torch.randn(3, 4, 8)
K_test = torch.randn(3, 5, 8)
V_test = torch.randn(3, 5, 6)

output_test, weights_test = attention(
    Q_test,
    K_test,
    V_test,
    mask=None
)

# 1. Output shape
assert output_test.shape == (3, 4, 6)

# 2. Attention weight shape
assert weights_test.shape == (3, 4, 5)

# 3. 마지막 차원 합 = 1
assert torch.allclose(
    weights_test.sum(dim=-1),
    torch.ones(3, 4),
    atol=1e-6
)

# 4. 여러 batch 정상 처리
assert output_test.size(0) == 3

# 5. mask=None 정상 처리
assert torch.isfinite(output_test).all()

# ---------------------------
# Mask 입력
# ---------------------------

Q_mask = torch.randn(1, 2, 4)
K_mask = torch.randn(1, 3, 4)
V_mask = torch.randn(1, 3, 5)

mask_test = torch.tensor([
    [
        [True, True, False],
        [True, True, True]
    ]
])

output_mask, weights_mask = attention(
    Q_mask,
    K_mask,
    V_mask,
    mask=mask_test
)

# 6. Mask된 위치의 weight = 0
assert torch.isclose(
    weights_mask[0, 0, 2],
    torch.tensor(0.0),
    atol=1e-7
)

# 7. Mask가 있어도 weight 합 = 1
assert torch.allclose(
    weights_mask.sum(dim=-1),
    torch.ones(1, 2),
    atol=1e-6
)

print("모든 Scaled Dot-Product Attention 테스트 통과")

모든 Scaled Dot-Product Attention 테스트 통과


In [ ]:
# 파일로 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/attention.py

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class ScaledDotProductAttention(nn.Module):
    """
    Attention(Q, K, V)
        = softmax(QK^T / sqrt(d_k)) V
    """

    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q:
                (..., query_len, d_k)

            K:
                (..., key_len, d_k)

            V:
                (..., key_len, d_v)

            mask:
                Attention mask broadcastable to
                (..., query_len, key_len).

                True  = attention 허용
                False = attention 차단

        Returns:
            output:
                (..., query_len, d_v)

            attention_weights:
                (..., query_len, key_len)
        """

        if Q.size(-1) != K.size(-1):
            raise ValueError(
                f"Q와 K의 d_k가 같아야 합니다. "
                f"Q: {Q.size(-1)}, K: {K.size(-1)}"
            )

        if K.size(-2) != V.size(-2):
            raise ValueError(
                f"K와 V의 sequence 길이가 같아야 합니다. "
                f"K: {K.size(-2)}, V: {V.size(-2)}"
            )

        d_k = Q.size(-1)

        # QK^T
        scores = Q @ K.transpose(-2, -1)

        # Scaling
        scores = scores / math.sqrt(d_k)

        # Mask
        if mask is not None:
            mask = mask.to(dtype=torch.bool, device=scores.device)
            scores = scores.masked_fill(~mask, float("-inf"))

        # Attention probability
        attention_weights = F.softmax(scores, dim=-1)

        # Weighted sum of V
        output = attention_weights @ V

        return output, attention_weights

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/attention.py
